# Bilişsel Performans Skoru: Optuna ile CatBoost Optimizasyonu
Gelişmiş özellik mühendisliği ve Ensembling adımları her zaman başarı getirmez. Özellikle CatBoost, kendi içindeki kategorik değişken işleme (native categorical handling) algoritmasıyla çok güçlüdür. LightGBM ve XGBoost'u işin içine katmak için uyguladığımız `LabelEncoder` adımı, CatBoost'un bu doğal gücünü zayıflatmış olabilir.

Bu notebook'ta, en iyi performansı gösteren **CatBoost tabanlı K-Fold** yapımıza geri dönüyor ve parametreleri ezbere vermek yerine **Optuna** kullanarak verinize en uygun `depth`, `learning_rate` ve `l2_leaf_reg` değerlerini buluyoruz.

In [1]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 1.0 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [optuna]2m5/6 [optuna]emy]


In [2]:
import pandas as pd
import numpy as np
import optuna
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

In [3]:
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test_x.csv")
sample = pd.read_csv("../data/sample_submission.csv")

target = "bilissel_performans_skoru"
X = train.drop(target, axis=1)
y = train[target]

In [4]:
cat_features = X.select_dtypes(include=["object"]).columns.tolist()

for col in cat_features:
    X[col] = X[col].fillna("missing")
    test[col] = test[col].fillna("missing")

In [ ]:
def objective(trial):
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    oof_preds = np.zeros(len(train))

    params = {
        "iterations": trial.suggest_int("iterations", 500, 2000),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.1, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-1, 10.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "loss_function": "RMSE",
        "random_seed": 42,
        "verbose": False,
        "early_stopping_rounds": 50
    }
    
    for train_idx, valid_idx in kf.split(X, y):
        X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
        X_va, y_va = X.iloc[valid_idx], y.iloc[valid_idx]
        
        model = CatBoostRegressor(**params)
        model.fit(
            X_tr, y_tr, 
            eval_set=(X_va, y_va), 
            cat_features=cat_features, 
            use_best_model=True
        )
        
        oof_preds[valid_idx] = model.predict(X_va)
        
    rmse = np.sqrt(mean_squared_error(y, oof_preds))
    return rmse

In [ ]:

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20)

print("\nEn İyi Parametreler:", study.best_params)
print("En İyi OOF RMSE Skoru:", study.best_value)

[I 2026-05-10 16:16:57,384] A new study created in memory with name: no-name-311365de-ac29-4f92-b4fc-5ef17045dfc3
[I 2026-05-10 16:17:29,193] Trial 0 finished with value: 1.3823809428148648 and parameters: {'iterations': 1739, 'learning_rate': 0.0015056123312038139, 'depth': 4, 'l2_leaf_reg': 6.292506058524498, 'bagging_temperature': 0.5140540117305092}. Best is trial 0 with value: 1.3823809428148648.
[I 2026-05-10 16:18:10,302] Trial 1 finished with value: 1.2170654723104621 and parameters: {'iterations': 1178, 'learning_rate': 0.017843282251399434, 'depth': 7, 'l2_leaf_reg': 0.10250484133736598, 'bagging_temperature': 0.26979334777129305}. Best is trial 1 with value: 1.2170654723104621.
[I 2026-05-10 16:18:54,805] Trial 2 finished with value: 1.2369024176174293 and parameters: {'iterations': 1150, 'learning_rate': 0.005124103988499018, 'depth': 7, 'l2_leaf_reg': 4.368533232269884, 'bagging_temperature': 0.7153957956241835}. Best is trial 1 with value: 1.2170654723104621.
[I 2026-05-1


En İyi Parametreler: {'iterations': 1319, 'learning_rate': 0.020457517889251136, 'depth': 5, 'l2_leaf_reg': 0.38785425516589167, 'bagging_temperature': 0.18984783034168573}
En İyi OOF RMSE Skoru: 1.2165008434126061


In [ ]:
print("En iyi parametrelerle final model eğitiliyor...")

best_params = study.best_params
best_params["loss_function"] = "RMSE"
best_params["random_seed"] = 42
best_params["verbose"] = 200
best_params["early_stopping_rounds"] = 100

kf = KFold(n_splits=5, shuffle=True, random_state=42)
test_preds = np.zeros(len(test))

for fold, (train_idx, valid_idx) in enumerate(kf.split(X, y)):
    print(f"\n--- Fold {fold+1} ---")
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[valid_idx], y.iloc[valid_idx]
    
    model = CatBoostRegressor(**best_params)
    model.fit(
        X_tr, y_tr, 
        eval_set=(X_va, y_va), 
        cat_features=cat_features, 
        use_best_model=True
    )
    
    test_preds += model.predict(test) / kf.n_splits

test_preds = test_preds.clip(0, 10)

En iyi parametrelerle final model eğitiliyor...

--- Fold 1 ---
0:	learn: 2.2076508	test: 2.2185792	best: 2.2185792 (0)	total: 6.71ms	remaining: 8.84s
200:	learn: 1.2764876	test: 1.2847389	best: 1.2847389 (200)	total: 1.13s	remaining: 6.3s
400:	learn: 1.2268227	test: 1.2410048	best: 1.2410048 (400)	total: 2.38s	remaining: 5.46s
600:	learn: 1.2114254	test: 1.2299269	best: 1.2299251 (599)	total: 3.65s	remaining: 4.37s
800:	learn: 1.2017343	test: 1.2249260	best: 1.2249260 (800)	total: 4.82s	remaining: 3.12s
1000:	learn: 1.1948225	test: 1.2227816	best: 1.2227816 (1000)	total: 6s	remaining: 1.91s
1200:	learn: 1.1891571	test: 1.2216662	best: 1.2216582 (1199)	total: 7.15s	remaining: 702ms
1318:	learn: 1.1862926	test: 1.2213899	best: 1.2213635 (1302)	total: 7.82s	remaining: 0us

bestTest = 1.221363489
bestIteration = 1302

Shrink model to first 1303 iterations.

--- Fold 2 ---
0:	learn: 2.2114121	test: 2.2025868	best: 2.2025868 (0)	total: 7.26ms	remaining: 9.57s
200:	learn: 1.2780506	test: 1.2

In [8]:
submission = pd.DataFrame({
    "id": test["id"],
    "bilissel_performans_skoru": test_preds
})

submission.to_csv("../submissions/sub_optuna.csv", index=False)
submission.head()

,id,bilissel_performans_skoru
0,1,6.038576
1,2,6.762190
2,3,3.049011
3,4,7.151032
4,5,3.720385
